In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_03 — Silver Cleaning
# MAGIC **Cleans Bronze synthetic tables → enforces constraints → computes aggregated features → writes to `xscore.silver`**
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Reads the three Bronze synthetic tables
# MAGIC - Validates data: removes nulls, enforces value ranges, deduplicates
# MAGIC - Adds CHECK constraints on critical columns (score must be 0–100, etc.)
# MAGIC - Computes per-user aggregated features from the bill and UPI time-series
# MAGIC - Joins everything into a single Silver user table
# MAGIC - Enables CDF (Change Data Feed) on Silver tables for downstream triggers
# MAGIC - Writes to `xscore.silver`
# MAGIC
# MAGIC **Depends on:** NB_02 complete  
# MAGIC **Runtime:** ~6 minutes  
# MAGIC **Next:** NB_04_gold_feature_store

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Setup

# COMMAND ----------

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("✓ Setup complete")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Clean synthetic_profiles

# COMMAND ----------

profiles_bronze = spark.table("xscore.bronze.synthetic_profiles")

print(f"Bronze profiles: {profiles_bronze.count():,} rows")
print(f"Columns: {len(profiles_bronze.columns)}")

# ── Validation rules ─────────────────────────────────────────
# 1. user_id must not be null
# 2. income_monthly must be > 0
# 3. bill_ontime_rate_24m must be between 0 and 1
# 4. savings_ratio must be between -1 and 1
# 5. default_label must be 0 or 1
# 6. No duplicate user_ids

before = profiles_bronze.count()

profiles_clean = (profiles_bronze
    # Rule 1: no null user_id or segment
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("segment").isNotNull())
    # Rule 2: income must be positive
    .filter(F.col("income_monthly") > 0)
    # Rule 3: bill rate in valid range
    .filter(F.col("bill_ontime_rate_24m").between(0.0, 1.0))
    # Rule 4: savings ratio in valid range
    .filter(F.col("savings_ratio").between(-1.0, 1.0))
    # Rule 5: binary label
    .filter(F.col("default_label").isin(0, 1))
    # Rule 6: deduplicate on user_id (keep first)
    .dropDuplicates(["user_id"])
)

after = profiles_clean.count()
dropped = before - after
print(f"\nRows before cleaning : {before:,}")
print(f"Rows after cleaning  : {after:,}")
print(f"Rows dropped         : {dropped:,}  ({dropped/before:.2%})")

# Add a cleaned timestamp
profiles_clean = profiles_clean.withColumn(
    "silver_processed_at", F.current_timestamp()
)

# Write to Silver with CDF enabled
(profiles_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable("xscore.silver.user_profiles"))

print(f"\n✓ Written → xscore.silver.user_profiles")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Add CHECK constraints to Silver profiles table

# COMMAND ----------

# MAGIC %sql
# MAGIC -- CHECK constraints enforce data quality at the Delta layer.
# MAGIC -- Any future MERGE or INSERT that violates these will be rejected.
# MAGIC -- This protects the model from ever training on bad data.

# COMMAND ----------

# MAGIC %sql
# MAGIC ALTER TABLE xscore.silver.user_profiles
# MAGIC ADD CONSTRAINT valid_income
# MAGIC CHECK (income_monthly > 0 AND income_monthly < 500000);

# COMMAND ----------

# MAGIC %sql
# MAGIC ALTER TABLE xscore.silver.user_profiles
# MAGIC ADD CONSTRAINT valid_bill_rate
# MAGIC CHECK (bill_ontime_rate_24m >= 0 AND bill_ontime_rate_24m <= 1);

# COMMAND ----------

# MAGIC %sql
# MAGIC ALTER TABLE xscore.silver.user_profiles
# MAGIC ADD CONSTRAINT valid_default_label
# MAGIC CHECK (default_label = 0 OR default_label = 1);

# COMMAND ----------

# MAGIC %sql
# MAGIC ALTER TABLE xscore.silver.user_profiles
# MAGIC ADD CONSTRAINT valid_segment
# MAGIC CHECK (segment IN ('shg_woman', 'gig_worker', 'kirana_owner',
# MAGIC                    'salaried_informal', 'rural_farmer'));

# COMMAND ----------

print("✓ CHECK constraints added to xscore.silver.user_profiles")
print("  Constraints enforced:")
print("    - income_monthly: 0 < x < 500,000")
print("    - bill_ontime_rate_24m: 0 ≤ x ≤ 1")
print("    - default_label: 0 or 1 only")
print("    - segment: one of 5 valid values")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Clean bill payment history

# COMMAND ----------

bills_bronze = spark.table("xscore.bronze.synthetic_bills")
print(f"Bronze bills: {bills_bronze.count():,} rows")

bills_clean = (bills_bronze
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("amount") > 0)
    .filter(F.col("amount") < 50000)          # cap at ₹50K per bill
    .filter(F.col("paid_on_time").isin(0, 1))
    .filter(F.col("days_late").between(0, 90))
    .filter(F.col("month").between(1, 24))
    .filter(F.col("bill_type").isin(
        "electricity", "mobile", "water", "dth"
    ))
    .dropDuplicates(["user_id", "month", "bill_type"])
)

print(f"Bills after cleaning: {bills_clean.count():,} rows")

(bills_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("bill_type")
    .saveAsTable("xscore.silver.bill_payments"))

print("✓ Written → xscore.silver.bill_payments")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Clean UPI transactions

# COMMAND ----------

upi_bronze = spark.table("xscore.bronze.synthetic_upi_txns")
print(f"Bronze UPI: {upi_bronze.count():,} rows")

upi_clean = (upi_bronze
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("amount") > 0)
    .filter(F.col("amount") < 200000)         # cap at ₹2L per transaction
    .filter(F.col("status").isin("SUCCESS", "FAILED"))
    .filter(F.col("month").between(1, 12))
    .dropDuplicates(["user_id", "month", "amount", "category"])
)

print(f"UPI after cleaning: {upi_clean.count():,} rows")

(upi_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("status")
    .saveAsTable("xscore.silver.upi_transactions"))

print("✓ Written → xscore.silver.upi_transactions")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Compute Pillar 1 features from bill history

# COMMAND ----------

# Read cleaned bills
bills = spark.table("xscore.silver.bill_payments")

pillar1_features = bills.groupBy("user_id").agg(

    # Core timeliness
    F.round(F.mean("paid_on_time"), 4)
     .alias("p1_bill_ontime_rate"),

    F.round(F.mean("days_late"), 2)
     .alias("p1_avg_days_late"),

    F.round(F.max("days_late").cast("double"), 1)
     .alias("p1_max_days_late"),

    # Severe late = more than 30 days
    F.round(
        F.sum(F.when(F.col("days_late") > 30, 1).otherwise(0)).cast("double")
        / F.count("*"), 4
    ).alias("p1_severe_late_rate"),

    # Bill type diversity — how many different bill types they pay
    F.countDistinct("bill_type").cast("double")
     .alias("p1_bill_type_diversity"),

    # Total bills paid (signals committed regular outgoings)
    F.round(F.sum("amount"), 2)
     .alias("p1_total_bills_24m"),

    # Payment trend: recent 6 months vs first 6 months
    # Positive = improving, negative = getting worse
    F.round(
        F.mean(F.when(F.col("month") >= 19,
               F.col("paid_on_time").cast("double"))),
        4
    ).alias("p1_ontime_recent_6m"),

    F.round(
        F.mean(F.when(F.col("month") <= 6,
               F.col("paid_on_time").cast("double"))),
        4
    ).alias("p1_ontime_first_6m"),

).withColumn(
    "p1_payment_trend",
    F.round(F.col("p1_ontime_recent_6m") - F.col("p1_ontime_first_6m"), 4)
)

print(f"✓ Pillar 1 features computed: {pillar1_features.count():,} users")
print(f"  Features: {[c for c in pillar1_features.columns if c != 'user_id']}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Compute Pillar 2 features from UPI history

# COMMAND ----------

upi = spark.table("xscore.silver.upi_transactions")

pillar2_features = upi.groupBy("user_id").agg(

    # Transaction volume (per month average)
    F.round((F.count("*") / F.lit(12.0)), 2)
     .alias("p2_upi_txn_per_month"),

    # Amount statistics
    F.round(F.mean("amount"), 2)
     .alias("p2_avg_txn_amount"),

    F.round(F.stddev("amount"), 2)
     .alias("p2_std_txn_amount"),

    # Failure rate — proxy for insufficient funds events
    F.round(
        F.sum(F.when(F.col("status") == "FAILED", 1).otherwise(0)).cast("double")
        / F.count("*"), 4
    ).alias("p2_failure_rate"),

    # Merchant diversity — how many different categories
    F.countDistinct("category").cast("double")
     .alias("p2_merchant_diversity"),

    # Total spend on successful transactions
    F.round(
        F.sum(F.when(F.col("status") == "SUCCESS", F.col("amount")).otherwise(0)),
        2
    ).alias("p2_total_spend_12m"),

).withColumn(
    # Coefficient of variation: std/mean — lower = more consistent spending
    "p2_txn_cv",
    F.round(F.col("p2_std_txn_amount") / (F.col("p2_avg_txn_amount") + 0.01), 4)
)

print(f"✓ Pillar 2 features computed: {pillar2_features.count():,} users")
print(f"  Features: {[c for c in pillar2_features.columns if c != 'user_id']}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Join all pillars into Silver user_features table

# COMMAND ----------

# Start with cleaned profiles (Pillars 3-6 already in there from generation)
profiles = spark.table("xscore.silver.user_profiles")

# Join computed Pillar 1 and Pillar 2 features
silver_features = (profiles
    .join(pillar1_features, on="user_id", how="left")
    .join(pillar2_features, on="user_id", how="left")
    .fillna(0.0)   # users with no bills/UPI data get zeros
)

# Add 4 derived composite features that cross multiple pillars
silver_features = silver_features.withColumns({

    # Income vs committed bills ratio (Pillar 4 × Pillar 1)
    # Higher = more income relative to bill obligations = lower stress
    "derived_income_to_bills_ratio": F.round(
        F.col("income_monthly") / (F.col("p1_total_bills_24m") / 24.0 + 1.0),
        2
    ),

    # Digital engagement score (Pillar 2 composite, 0-100 scale)
    "derived_digital_engagement": F.round(
        F.col("p2_upi_txn_per_month") * 0.4
        + F.col("p2_merchant_diversity") * 10.0
        + (1.0 - F.col("p2_failure_rate")) * 50.0,
        2
    ),

    # Formal economy participation score (Pillars 4 + 5, 0-100 scale)
    "derived_formal_economy_score": F.round(
        F.col("itr_filed") * 40.0
        + F.col("gst_registered") * 30.0
        + F.col("jan_dhan_active") * 15.0
        + F.col("shg_member") * 15.0,
        2
    ),

    # Asset stability score (Pillar 3, 0-100 scale)
    "derived_asset_score": F.round(
        F.col("owns_land") * 40.0
        + F.col("owns_vehicle") * 20.0
        + F.col("has_fd_or_rd") * 25.0
        + F.least(F.col("bank_vintage_months") / 2.0, F.lit(15.0)),
        2
    ),
})

row_count = silver_features.count()
col_count = len(silver_features.columns)

print(f"Silver user_features: {row_count:,} rows × {col_count} columns")

# Write with CDF enabled — NB_09 (nightly job) will use CDF to detect changes
(silver_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("segment")
    .saveAsTable("xscore.silver.user_features"))

print("✓ Written → xscore.silver.user_features")
print("  CDF enabled — downstream jobs will detect row-level changes")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Verify Silver layer

# COMMAND ----------

# MAGIC %sql
# MAGIC SHOW TABLES IN xscore.silver;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Quick sanity check: average features by segment
# MAGIC -- Income, bill payment rate, and default rate should
# MAGIC -- increase in the order: rural_farmer < shg_woman < gig_worker
# MAGIC --                        < salaried_informal < kirana_owner
# MAGIC SELECT
# MAGIC   segment,
# MAGIC   COUNT(*)                                   AS users,
# MAGIC   ROUND(AVG(income_monthly), 0)              AS avg_income,
# MAGIC   ROUND(AVG(bill_ontime_rate_24m), 3)        AS avg_bill_ontime,
# MAGIC   ROUND(AVG(p1_bill_ontime_rate), 3)         AS avg_p1_ontime,
# MAGIC   ROUND(AVG(p2_upi_txn_per_month), 1)        AS avg_upi_txn,
# MAGIC   ROUND(AVG(derived_formal_economy_score), 1)AS avg_formal_score,
# MAGIC   ROUND(AVG(default_label), 3)               AS default_rate
# MAGIC FROM xscore.silver.user_features
# MAGIC GROUP BY segment
# MAGIC ORDER BY avg_income ASC;

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_03 SILVER CLEANING — COMPLETE")
print("=" * 60)

silver_tables = {
    "user_profiles" : "xscore.silver.user_profiles",
    "bill_payments" : "xscore.silver.bill_payments",
    "upi_transactions": "xscore.silver.upi_transactions",
    "user_features" : "xscore.silver.user_features",
}

for name, tbl in silver_tables.items():
    n = spark.table(tbl).count()
    print(f"  ✓  {name:<25} {n:>12,} rows")

print("=" * 60)
print()
print("  CHECK constraints active on user_profiles:")
print("    income_monthly · bill_ontime_rate_24m")
print("    default_label  · segment")
print()
print("  CDF enabled on all Silver tables:")
print("    NB_09 nightly job will detect only changed rows")
print()
print("  NEXT: Run NB_04_gold_feature_store")
print("  NB_04 reads silver.user_features and writes")
print("  the final model-ready Gold feature store.")
print("=" * 60)